# 07 · GRU 核心实现

目标：理解 `r,z,n` 三门、update 插值和 PyTorch 兼容的 reset-after 候选公式。

In [ ]:
import inspect, os
from pathlib import Path
if not (Path.cwd() / 'common').exists() and (Path.cwd().parent / 'common').exists():
    os.chdir(Path.cwd().parent)
import matplotlib.pyplot as plt
import numpy as np
import torch
from common.gradcheck import check_gradient, numerical_gradient
from gru.numpy_impl import sigmoid, init_parameters, step_forward, sequence_forward, sequence_backward
from gru.torch_impl import ManualGRUCell, ManualRecurrentLayer

## 1. reset-after 单步公式

输入投影和隐藏投影分别切成 `r,z,n`：

$$r=sigmoid(x_r+h_r); z=sigmoid(x_z+h_z)$$
$$n=tanh(x_n+r*h_n); h_t=(1-z)*n+z*h_{t-1}$$

关键点：reset gate 乘在隐藏候选投影之后；一般情况下它不能移到矩阵乘法之前。

In [ ]:
def notebook_gru_step(x_t, h_prev, params):
    input_projection = x_t @ params['weight_ih'].T + params['bias_ih']
    hidden_projection = h_prev @ params['weight_hh'].T + params['bias_hh']
    xr, xz, xn = np.split(input_projection, 3, axis=1)
    hr, hz, hn = np.split(hidden_projection, 3, axis=1)
    r, z = sigmoid(xr + hr), sigmoid(xz + hz)
    n = np.tanh(xn + r * hn)
    return (1.0 - z) * n + z * h_prev

rng = np.random.default_rng(42)
example_params = init_parameters(3, 4, rng)
x_np, h_np = rng.normal(size=(2, 3)), rng.normal(size=(2, 4))
expected_h, cache = step_forward(x_np, h_np, example_params)
np.testing.assert_allclose(notebook_gru_step(x_np, h_np, example_params), expected_h)
print('单步输出:', expected_h.shape, '| cache:', sorted(cache))

## 2. 分支 BPTT

最终插值首先把梯度分到 candidate、update gate 和旧状态直连；candidate 再分到输入候选、reset gate 和隐藏候选。`dh_prev` 必须合并所有这些路径。

In [ ]:
print(inspect.getsource(sequence_forward))
print(inspect.getsource(sequence_backward))

In [ ]:
rng = np.random.default_rng(7)
x = rng.normal(size=(2, 2, 2)).astype(np.float64)
h0 = rng.normal(size=(2, 2)).astype(np.float64)
params = init_parameters(2, 2, rng)
outputs, _, caches = sequence_forward(x, h0, params)
doutputs = rng.normal(size=outputs.shape)
dx, _, _ = sequence_backward(doutputs, caches)
def objective():
    current, _, _ = sequence_forward(x, h0, params)
    return float(np.sum(current * doutputs))
result = check_gradient('gru-dx', dx, numerical_gradient(objective, x))
print(result)
assert result.passed

## 3. PyTorch 对齐与 gate 行为

保持输入、隐藏投影分开，才能复现官方 reset-after 候选分支。

In [ ]:
print(inspect.getsource(ManualGRUCell.forward))
manual = ManualGRUCell(3, 4).double()
official = torch.nn.GRUCell(3, 4).double()
official.load_state_dict(manual.state_dict())
x_t = torch.randn(2, 3, dtype=torch.float64)
h_prev = torch.randn(2, 4, dtype=torch.float64)
torch.testing.assert_close(manual(x_t, h_prev), official(x_t, h_prev))
print('官方 GRUCell 前向对齐通过')

In [ ]:
torch.manual_seed(42)
cell = ManualGRUCell(3, 8)
sequence = torch.randn(1, 30, 3)
h = torch.zeros(1, 8)
reset_means, update_means = [], []
with torch.no_grad():
    for time in range(sequence.shape[1]):
        xi = sequence[:, time]
        xp = xi @ cell.weight_ih.T + cell.bias_ih
        hp = h @ cell.weight_hh.T + cell.bias_hh
        xr, xz, _ = xp.chunk(3, dim=1); hr, hz, _ = hp.chunk(3, dim=1)
        reset_means.append(torch.sigmoid(xr + hr).mean().item())
        update_means.append(torch.sigmoid(xz + hz).mean().item())
        h = cell(xi, h)
plt.plot(reset_means, label='reset mean'); plt.plot(update_means, label='update mean')
plt.ylim(0, 1); plt.xlabel('time index'); plt.ylabel('gate mean'); plt.legend();
plt.title('GRU gate statistics'); plt.show()

## 4. 回忆区

凭记忆补出 reset-after 公式，特别注意 reset gate 的乘法位置。

In [ ]:
RUN_RECALL_TESTS = False
def recall_gru_step(x_t, h_prev, params):
    # TODO: 返回 h_t
    pass
if RUN_RECALL_TESTS:
    recalled = recall_gru_step(x_np, h_np, example_params)
    np.testing.assert_allclose(recalled, notebook_gru_step(x_np, h_np, example_params))
    print('回忆测试通过')

### 复盘问题

1. update gate 接近 1 时保留哪一项？
2. reset gate 影响哪些分支？
3. 为什么 `W(r*h)` 与 `r*(Wh+b)` 通常不同？
4. `dh_prev` 有哪些来源？